In [ ]:
import networkx as nx
import numpy as np
#from bngenerator import *
from bngenerator import *
from matplotlib import pyplot as plt
from pgmpy.readwrite.XMLBeliefNetwork import XBNReader, XBNWriter
from pgmpy.readwrite import BIFWriter, BIFReader

#writer = BIFWriter(model)
#writer.write_bif("model.bif")
#my_dag = BIFReader("model.bif").get_model()

In [2]:
gen = BayesianNetworkGenerator(
        n_nodes=60,
        max_induced_width=12,
        max_degree=20,
        n_iterations=1000
    )

In [3]:
dag = gen.generate().dag

In [6]:
INDUCED_WIDTH = [2, 6, 12] #(SPARSE, MEDIUM, DENSE)
CPD_ALPHA = [0.2, 5] # 0.2 = deterministic, 5 = "fuzzy"
NUM_NODES = [10, 20, 30, 40, 50, 60, 80, 100, 150, 200] # H_size = num_nodes * 0.25, following Chen et al.

In [13]:
from pgmpy.models import BayesianNetwork
from pgmpy.factors.discrete import TabularCPD

def generate_controlled_binary_bn(num_nodes=20, induced_width=2, cpd_alpha=5):
    """
    Generates a random binary Bayesian Network with controlled parameters.
    cpd_alpha < 1.0 creates deterministic/rigid networks (closer to 0 or 1).
    cpd_alpha > 1.0 creates fuzzy/uncertain networks (closer to 0.5).
    """

    generator = BayesianNetworkGenerator(n_nodes=num_nodes, 
                                         max_induced_width=induced_width, 
                                         max_degree=20, 
                                         n_iterations=2_000)
    
    dag = generator.generate()
    dag = dag.dag
    # Convert to pgmpy format (using string names to avoid int indexing bugs!)
    edges = [(f"X{u}", f"X{v}") for u, v in dag.edges()]
    bn = BayesianNetwork(edges)
    
    # Add isolated nodes if any exist
    for i in range(num_nodes):
        if f"X{i}" not in bn.nodes():
            bn.add_node(f"X{i}")

    # 2. Populate CPDs using the controlled Beta distribution
    for node in bn.nodes():
        parents = list(bn.get_parents(node))
        num_parents = len(parents)
        num_parent_states = 2 ** num_parents
        
        # Draw probabilities from Beta(alpha, alpha)
        # E.g., if alpha=0.1, p_true will mostly be ~0.99 or ~0.01
        p_true = np.random.beta(cpd_alpha, cpd_alpha, size=num_parent_states)
        p_false = 1 - p_true
        
        # TabularCPD expects values as a list of lists: [[P(False)], [P(True)]]
        cpd_values = [p_false.tolist(), p_true.tolist()]
        
        cpd = TabularCPD(
            variable=node,
            variable_card=2,
            values=cpd_values,
            evidence=parents if num_parents > 0 else None,
            evidence_card=[2] * num_parents if num_parents > 0 else None,
            state_names={node: [0, 1], **{p: [0, 1] for p in parents}}
        )
        bn.add_cpds(cpd)
        
    assert bn.check_model()
    return bn

In [35]:
import os
def generate_and_save_bns():
    """
    Iterates through the specified parameters, generates BNs, 
    and saves them to uniquely named BIF files.
    """
    INDUCED_WIDTH = [2, 6, 12] # (SPARSE, MEDIUM, DENSE)
    CPD_ALPHA = [0.2, 5]       # 0.2 = deterministic, 5 = "fuzzy"
    NUM_NODES = [10, 20, 30, 40, 50, 60, 80, 100, 150, 200] 
    
    output_dir = "generated_bif_files"
    os.makedirs(output_dir, exist_ok=True)
    
    for n in NUM_NODES:
        for w in INDUCED_WIDTH:
            for a in CPD_ALPHA:
                print(f"Generating BN: {n} nodes | width {w} | alpha {a}...")
                
                try:
                    # Generate the network
                    bn = generate_controlled_binary_bn(num_nodes=n, induced_width=w, cpd_alpha=a)
                    
                    # Create descriptive labels for the filename
                    density_label = "sparse" if w == 2 else "medium" if w == 6 else "dense"
                    fuzziness_label = "det" if a == 0.2 else "fuzzy"
                    
                    # Format: bn_n20_w2-sparse_a0.2-det.bif
                    filename = f"bn_n{n}_w{w}_{density_label}_{fuzziness_label}.bif"
                    filepath = os.path.join(output_dir, filename)
                    
                    # Save to BIF format
                    writer = BIFWriter(bn)
                    writer.write_bif(filepath)
                    
                    print(f"  -> Successfully saved to {filepath}")
                    
                except Exception as e:
                    print(f"  -> [ERROR] Failed to generate/save BN for parameters (n={n}, w={w}, a={a}): {e}")


In [36]:
generate_and_save_bns()

Generating BN: 10 nodes | width 2 | alpha 0.2...
  -> Successfully saved to generated_bif_files/bn_n10_w2_sparse_det.bif
Generating BN: 10 nodes | width 2 | alpha 5...
  -> Successfully saved to generated_bif_files/bn_n10_w2_sparse_fuzzy.bif
Generating BN: 10 nodes | width 6 | alpha 0.2...
  -> Successfully saved to generated_bif_files/bn_n10_w6_medium_det.bif
Generating BN: 10 nodes | width 6 | alpha 5...
  -> Successfully saved to generated_bif_files/bn_n10_w6_medium_fuzzy.bif
Generating BN: 10 nodes | width 12 | alpha 0.2...
  -> Successfully saved to generated_bif_files/bn_n10_w12_dense_det.bif
Generating BN: 10 nodes | width 12 | alpha 5...
  -> Successfully saved to generated_bif_files/bn_n10_w12_dense_fuzzy.bif
Generating BN: 20 nodes | width 2 | alpha 0.2...
  -> Successfully saved to generated_bif_files/bn_n20_w2_sparse_det.bif
Generating BN: 20 nodes | width 2 | alpha 5...
  -> Successfully saved to generated_bif_files/bn_n20_w2_sparse_fuzzy.bif
Generating BN: 20 nodes | widt